# Sprint 3: Experimentación y Ajuste de Hiperparámetros

Este notebook ejecuta los experimentos del Sprint 3:

1. **Grid Search** de hiperparámetros
2. **Validación Cruzada** (k-fold)
3. **Análisis de Convergencia**
4. **Análisis de Sensibilidad**

## Objetivo
Encontrar la configuración óptima del Algoritmo Genético y validar su robustez.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from omnievo import (
    DataGenerator,
    GeneticOptimizer,
    grid_search,
    cross_validate,
    sensitivity_analysis,
    analyze_convergence,
    run_full_experiment,
    compare_baselines,
    plot_convergence,
    plot_weights,
)

# Configuración de plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Librerías cargadas correctamente")

## 1. Preparación de Datos

In [ ]:
# Generar dataset más grande para experimentos robustos
generator = DataGenerator(n_users=2000, random_state=42)
df = generator.generate()
channels = generator.get_channel_names()

X = df[channels].values
y = df['LTV_real'].values

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Dataset: {len(df)} usuarios")
print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Canales: {channels}")

## 2. Grid Search de Hiperparámetros

Buscamos la mejor combinación de:
- **population_size**: Tamaño de la población
- **generations**: Número de generaciones
- **cxpb**: Probabilidad de cruzamiento
- **mutpb**: Probabilidad de mutación

In [ ]:
# Definir grid de búsqueda
param_grid = {
    "population_size": [30, 50, 100],
    "generations": [50, 100, 150],
    "cxpb": [0.6, 0.7, 0.8],
    "mutpb": [0.1, 0.15, 0.2],
}

total_configs = np.prod([len(v) for v in param_grid.values()])
print(f"Total de configuraciones a evaluar: {total_configs}")

In [ ]:
# Ejecutar Grid Search (puede tomar varios minutos)
gs_result = grid_search(
    X_train, y_train,
    X_test, y_test,
    param_grid=param_grid,
    n_runs=1,
    verbose=True,
)

In [ ]:
# Mostrar mejores configuraciones
print("\nTop 10 Configuraciones:")
print(gs_result.results_df.head(10))

In [ ]:
# Mejor configuración
print(f"\nMejor Configuración:")
print(f"  Parámetros: {gs_result.best_params}")
print(f"  RMSE Test: {gs_result.best_result.rmse_test:.4f}")
print(f"  Pearson: {gs_result.best_result.pearson_test:.4f}")

## 3. Análisis de Sensibilidad

Evaluamos cómo afecta cada hiperparámetro al rendimiento.

In [ ]:
# Parámetros base (los mejores encontrados o defaults razonables)
base_params = {
    "population_size": 50,
    "generations": 100,
    "cxpb": 0.7,
    "mutpb": 0.15,
}

# Sensibilidad a población
sens_pop = sensitivity_analysis(
    X_train, y_train, X_test, y_test,
    base_params=base_params,
    param_name="population_size",
    param_values=[20, 30, 50, 75, 100, 150],
    n_runs=3,
)

In [ ]:
# Sensibilidad a generaciones
sens_gen = sensitivity_analysis(
    X_train, y_train, X_test, y_test,
    base_params=base_params,
    param_name="generations",
    param_values=[25, 50, 75, 100, 150, 200],
    n_runs=3,
)

In [ ]:
# Sensibilidad a mutación
sens_mut = sensitivity_analysis(
    X_train, y_train, X_test, y_test,
    base_params=base_params,
    param_name="mutpb",
    param_values=[0.05, 0.1, 0.15, 0.2, 0.25, 0.3],
    n_runs=3,
)

In [ ]:
# Visualizar sensibilidad
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Población
axes[0].errorbar(sens_pop['population_size'], sens_pop['rmse_mean'], 
                 yerr=sens_pop['rmse_std'], marker='o', capsize=5)
axes[0].set_xlabel('Tamaño de Población')
axes[0].set_ylabel('RMSE')
axes[0].set_title('Sensibilidad a Población')

# Generaciones
axes[1].errorbar(sens_gen['generations'], sens_gen['rmse_mean'],
                 yerr=sens_gen['rmse_std'], marker='o', capsize=5, color='orange')
axes[1].set_xlabel('Generaciones')
axes[1].set_ylabel('RMSE')
axes[1].set_title('Sensibilidad a Generaciones')

# Mutación
axes[2].errorbar(sens_mut['mutpb'], sens_mut['rmse_mean'],
                 yerr=sens_mut['rmse_std'], marker='o', capsize=5, color='green')
axes[2].set_xlabel('Probabilidad de Mutación')
axes[2].set_ylabel('RMSE')
axes[2].set_title('Sensibilidad a Mutación')

plt.tight_layout()
plt.show()

## 4. Validación Cruzada (5-Fold)

Validamos la robustez del modelo con los mejores hiperparámetros.

In [ ]:
# Usar mejores parámetros del grid search
best_params = gs_result.best_params
print(f"Parámetros para CV: {best_params}")

# Ejecutar validación cruzada
cv_result = cross_validate(
    X, y,
    k_folds=5,
    optimizer_params=best_params,
    verbose=True,
)

In [ ]:
# Visualizar resultados por fold
folds_df = pd.DataFrame(cv_result['folds'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# RMSE por fold
axes[0].bar(folds_df['fold'], folds_df['rmse'], color='steelblue', alpha=0.8)
axes[0].axhline(cv_result['rmse_mean'], color='red', linestyle='--', label=f"Media: {cv_result['rmse_mean']:.2f}")
axes[0].fill_between([0.5, 5.5], 
                     cv_result['rmse_mean'] - cv_result['rmse_std'],
                     cv_result['rmse_mean'] + cv_result['rmse_std'],
                     alpha=0.2, color='red')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('RMSE')
axes[0].set_title('RMSE por Fold')
axes[0].legend()

# Pearson por fold
axes[1].bar(folds_df['fold'], folds_df['pearson'], color='forestgreen', alpha=0.8)
axes[1].axhline(cv_result['pearson_mean'], color='red', linestyle='--', label=f"Media: {cv_result['pearson_mean']:.3f}")
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('Correlación Pearson')
axes[1].set_title('Pearson por Fold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nResultados CV:")
print(f"  RMSE: {cv_result['rmse_mean']:.4f} ± {cv_result['rmse_std']:.4f}")
print(f"  Pearson: {cv_result['pearson_mean']:.4f} ± {cv_result['pearson_std']:.4f}")

In [ ]:
# Estabilidad de pesos entre folds
print("\nEstabilidad de Pesos (media ± std entre folds):")
for i, ch in enumerate(channels):
    w_mean = cv_result['weights_mean'][i]
    w_std = cv_result['weights_std'][i]
    print(f"  {ch:20s}: {w_mean:.4f} ± {w_std:.4f}")

## 5. Análisis de Convergencia

Estudiamos cómo converge el AG durante la optimización.

In [ ]:
# Ejecutar modelo con mejores parámetros y más generaciones
optimizer = GeneticOptimizer(
    **best_params,
    random_state=42,
    verbose=False,
)
result = optimizer.fit(X_train, y_train)

# Analizar convergencia
conv = analyze_convergence(result)

print("Análisis de Convergencia:")
print(f"  RMSE inicial: {conv['initial_rmse']:.4f}")
print(f"  RMSE final: {conv['final_rmse']:.4f}")
print(f"  Mejora total: {conv['improvement_pct']:.1f}%")
print(f"  Gen 90% mejora: {conv['gen_90_improvement']}")
print(f"  Convergió en gen: {conv['convergence_gen']}")
print(f"  Diversidad promedio: {conv['avg_diversity']:.4f}")
print(f"  Estabilidad final: {conv['final_stability']:.4f}")

In [ ]:
# Gráfica de convergencia detallada
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

generations = [s.generation for s in result.history]
best_rmse = [-s.best_fitness for s in result.history]
avg_rmse = [-s.avg_fitness for s in result.history]
std_fitness = [s.std_fitness for s in result.history]

# Plot 1: Convergencia
axes[0].plot(generations, best_rmse, label='Mejor RMSE', linewidth=2, color='#2ecc71')
axes[0].plot(generations, avg_rmse, label='RMSE Promedio', linewidth=2, color='#3498db', alpha=0.7)
axes[0].fill_between(generations, 
                     np.array(avg_rmse) - np.array(std_fitness),
                     np.array(avg_rmse) + np.array(std_fitness),
                     alpha=0.2, color='#3498db')
if conv['convergence_gen'] > 0:
    axes[0].axvline(conv['convergence_gen'], color='red', linestyle='--', 
                    label=f"Convergencia (gen {conv['convergence_gen']})")
axes[0].set_xlabel('Generación')
axes[0].set_ylabel('RMSE')
axes[0].set_title('Curva de Convergencia')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot 2: Diversidad (gap entre mejor y promedio)
diversity = [avg - best for avg, best in zip(avg_rmse, best_rmse)]
axes[1].plot(generations, diversity, linewidth=2, color='#9b59b6')
axes[1].set_xlabel('Generación')
axes[1].set_ylabel('Gap (Promedio - Mejor)')
axes[1].set_title('Diversidad de la Población')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Comparación Final con Baselines

In [ ]:
# Comparación final
from omnievo.fitness import predict_ltv

comparison = compare_baselines(X_test, y_test, ga_weights=result.best_weights)
print("\nComparación Final con Baselines (Test Set):")
print(comparison.to_string(index=False))

In [ ]:
# Pesos finales optimizados
print("\nPesos de Atribución Finales:")
print("=" * 50)
weights = result.best_weights
for ch, w in sorted(zip(channels, weights), key=lambda x: -x[1]):
    bar = "█" * int(w * 40)
    print(f"{ch:20s} {w:.4f} ({w*100:5.1f}%) {bar}")

## 7. Resumen Sprint 3

In [ ]:
print("=" * 70)
print("RESUMEN SPRINT 3: EXPERIMENTACIÓN")
print("=" * 70)

print("\n1. GRID SEARCH")
print(f"   Configuraciones evaluadas: {len(gs_result.results)}")
print(f"   Mejores parámetros: {gs_result.best_params}")

print("\n2. VALIDACIÓN CRUZADA (5-fold)")
print(f"   RMSE: {cv_result['rmse_mean']:.4f} ± {cv_result['rmse_std']:.4f}")
print(f"   Pearson: {cv_result['pearson_mean']:.4f} ± {cv_result['pearson_std']:.4f}")

print("\n3. CONVERGENCIA")
print(f"   Mejora total: {conv['improvement_pct']:.1f}%")
print(f"   Generación de convergencia: {conv['convergence_gen']}")

print("\n4. MEJORA VS BASELINES")
ga_rmse = comparison[comparison['Modelo'] == 'Algoritmo Genético']['RMSE'].values[0]
uni_rmse = comparison[comparison['Modelo'] == 'Uniforme (1/N)']['RMSE'].values[0]
mejora = (uni_rmse - ga_rmse) / uni_rmse * 100
print(f"   Mejora vs Uniforme: {mejora:.1f}%")

print("\n" + "=" * 70)
print("SPRINT 3 COMPLETADO ✓")
print("=" * 70)